In [32]:
### Document Structure

from langchain_core.documents import Document




In [33]:
doc = Document(
    page_content = "This is the content of the document.",
    metadata = {
        "source":"example.pdf",
        "pages": 10,
        "author":"Zayed Shah",
        "date_created":"2024-06-01"
    }
)

doc

Document(metadata={'source': 'example.pdf', 'pages': 10, 'author': 'Zayed Shah', 'date_created': '2024-06-01'}, page_content='This is the content of the document.')

In [34]:
## Create a simple txt file

import os 

os.makedirs("../data/text_files",exist_ok=True)

In [35]:
sample_texts = {
    "../data/text_files/python_intro.txt": """ 

    Python is a high-level, interpreted programming language known for its simplicity and readability. 
    It was created by Guido van Rossum and first released in 1991. 
    Python's design philosophy emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code compared to other programming languages.
    Some of the use cases of Python include web development, data analysis, artificial intelligence, scientific computing, automation, and more.
    Apart from its versatility, Python has a large and active community that contributes to a vast ecosystem of libraries and frameworks, making it a popular choice for developers across various domains.
    Python's ease of use and extensive libraries have made it a go-to language for beginners and experienced programmers alike, solidifying its position as one of the most widely used programming languages in the world.
    """,
    "../data/text_files/data_science.txt": """
    Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from structured and unstructured data.
    It involves statistics, machine learning, data visualization, and domain expertise to solve complex problems and make data-driven decisions.
    Data scientists analyze and interpret complex data to help organizations make informed decisions, identify trends, and gain a competitive edge in various industries such as finance, healthcare, marketing, and more.
    Data science combines techniques from various fields, including mathematics, computer science, and domain-specific knowledge, to extract valuable insights from data and drive innovation in today's data-driven world.
    Machine learning, a subset of data science, enables computers to learn from data and make predictions or decisions without being explicitly programmed, further enhancing the capabilities of data science in solving real-world problems.
    """
}

for filepath, content in sample_texts.items():
    with open(filepath,"w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [36]:
# Text Loader

# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader 


loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document  = loader.load()
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content=" \n\n    Python is a high-level, interpreted programming language known for its simplicity and readability. \n    It was created by Guido van Rossum and first released in 1991. \n    Python's design philosophy emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code compared to other programming languages.\n    Some of the use cases of Python include web development, data analysis, artificial intelligence, scientific computing, automation, and more.\n    Apart from its versatility, Python has a large and active community that contributes to a vast ecosystem of libraries and frameworks, making it a popular choice for developers across various domains.\n    Python's ease of use and extensive libraries have made it a go-to language for beginners and experienced programmers alike, solidifying its position as one of the most widely used programming languages

In [37]:
# Directory Loader

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob = "**/*.txt",  #pattern to match
    loader_cls = TextLoader, #loader class to use
    loader_kwargs={'encoding':"utf-8"},
    show_progress=False
)

print(dir_loader.load())

[Document(metadata={'source': '..\\data\\text_files\\data_science.txt'}, page_content="\n    Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from structured and unstructured data.\n    It involves statistics, machine learning, data visualization, and domain expertise to solve complex problems and make data-driven decisions.\n    Data scientists analyze and interpret complex data to help organizations make informed decisions, identify trends, and gain a competitive edge in various industries such as finance, healthcare, marketing, and more.\n    Data science combines techniques from various fields, including mathematics, computer science, and domain-specific knowledge, to extract valuable insights from data and drive innovation in today's data-driven world.\n    Machine learning, a subset of data science, enables computers to learn from data and make predictions or decisions without being explic

In [38]:

from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob = "**/*.pdf",  #pattern to match
    loader_cls = PyMuPDFLoader, #loader class to use
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Oracle Analytics Publisher', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\OfferLetter.pdf', 'file_path': '..\\data\\pdf\\OfferLetter.pdf', 'total_pages': 3, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'encryption': 'Standard V4 R4 128-bit AES', 'modDate': '', 'creationDate': '', 'page': 0}, page_content="Cognizant Technology Solutions India Private Limited, Ground Floor, SDB-1, Plot No H-4, SIPCOT IT \nPARK, Padur Post, Siruseri, Chengalpattu District - 603103, Tamil Nadu, India.\n \n \n \n \n \n \n \n26-Feb-2026\n \nCandidate ID: 36923329\n \nZayed Shah Syed\nB.Tech Computer Science\nKakatiya Institute of Technology and Science, Warangal\n \nDear Zayed Shah Syed,\n \nFurther to our Letter of Intent for the position of Programmer Analyst Trainee / Programmer Analyst aligned\nto the hiring category, we are pleased to offer you an internship with us at Cognizant office for a 

In [39]:
#  Splitting the pdf into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [40]:
chunks = split_documents(pdf_documents)
chunks

Split 3 documents into 12 chunks

Example chunk:
Content: Cognizant Technology Solutions India Private Limited, Ground Floor, SDB-1, Plot No H-4, SIPCOT IT 
PARK, Padur Post, Siruseri, Chengalpattu District - 603103, Tamil Nadu, India.
 
 
 
 
 
 
 
26-Feb-2...
Metadata: {'producer': 'Oracle Analytics Publisher', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\OfferLetter.pdf', 'file_path': '..\\data\\pdf\\OfferLetter.pdf', 'total_pages': 3, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'encryption': 'Standard V4 R4 128-bit AES', 'modDate': '', 'creationDate': '', 'page': 0}


[Document(metadata={'producer': 'Oracle Analytics Publisher', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\OfferLetter.pdf', 'file_path': '..\\data\\pdf\\OfferLetter.pdf', 'total_pages': 3, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'encryption': 'Standard V4 R4 128-bit AES', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Cognizant Technology Solutions India Private Limited, Ground Floor, SDB-1, Plot No H-4, SIPCOT IT \nPARK, Padur Post, Siruseri, Chengalpattu District - 603103, Tamil Nadu, India.\n \n \n \n \n \n \n \n26-Feb-2026\n \nCandidate ID: 36923329\n \nZayed Shah Syed\nB.Tech Computer Science\nKakatiya Institute of Technology and Science, Warangal\n \nDear Zayed Shah Syed,\n \nFurther to our Letter of Intent for the position of Programmer Analyst Trainee / Programmer Analyst aligned\nto the hiring category, we are pleased to offer you an internship with us at Cognizant office for a 

#  Embedding and VectorStore DB

In [41]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple 
from sklearn.metrics.pairwise import cosine_similarity


In [42]:
class EmbeddingManager:
    """ Handles document embedding generation using SenctenceTransfromer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize Embedding Manager
        """

        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """ Generate Embedding for a list of texts

            Args: lists of txt strings to embed

            Returns: numpy array of embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully with shape: {embeddings.shape}.")
        return embeddings
    

# Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

        

Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4251.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding Dimension: 384


C:\Users\syedz\AppData\Local\Temp\ipykernel_22044\1865086769.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")


#  Vector Store

In [43]:
class VectorStore:
    """Manages document embeddings in ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """ 
        Initialize the Vector Store
        Args:
            collection_name: Name of the ChromaDB collection to store embeddings
            persist_directory: Directory where the ChromaDB database will be persisted
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self.initialize_store()

    def initialize_store(self):
        """ Initialize ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client  = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata = {"description": "Collection of PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized successfully with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and embeddings to vector store
            Args:
                documents: List of document metadata or content to store
                embeddings: Corresponding list of embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids=[]
        metadatas = []
        document_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content) 
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_text,
                embeddings=embeddings_list
            )
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized successfully with collection: pdf_documents
Existing documents in collection: 60


In [44]:
#  Convert the text to embeddings

texts = [doc.page_content for doc in chunks]
# texts 

embeddings = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]

Embeddings generated successfully with shape: (12, 384).
Adding 12 documents to vector store...
Documents added successfully. Total documents in collection: 72


## Retrieval

In [45]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self,vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever 

        Args:
            vectorstore: Instance of the VectorStore class to retrieve documents from
            embedding_manager: Instance of the EmbeddingManager to generate query embeddings 
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -0.5) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on the query

        Args:
            query: The user query string to search for
            top_k: Number of top relevant documents to retrieve
        Returns:
            List of retrieved documents with metadata
        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k}...")
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids,documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score  >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })
            print(f"Retrieved {len(retrieved_docs)} relevant documents.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise
                        

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

            
           


In [46]:
results = rag_retriever.retrieve("Tell about the offer letter", score_threshold=-1.0)
for doc in results[:3]:  # Show first 3 results
    print(f"Similarity: {doc['similarity_score']:.3f}")
    print(f"Content: {doc['content'][:200]}...")
    print("---")

Retrieving documents for query: 'Tell about the offer letter' with top_k=5...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.26it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 5 relevant documents.
Similarity: -0.021
Content: your Internship Offer your Letter of Intent would also be revoked at the sole discretion of Cognizant.
 
Below are the mandatory documents to be submitted as part of your Background Verification:
• Yo...
---
Similarity: -0.021
Content: your Internship Offer your Letter of Intent would also be revoked at the sole discretion of Cognizant.
 
Below are the mandatory documents to be submitted as part of your Background Verification:
• Yo...
---
Similarity: -0.021
Content: your Internship Offer your Letter of Intent would also be revoked at the sole discretion of Cognizant.
 
Below are the mandatory documents to be submitted as part of your Background Verification:
• Yo...
---


## Generating LLM Output

In [47]:
from langchain_groq import ChatGroq 
import os
from dotenv import load_dotenv
load_dotenv()


groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

# Simple RAG Function

def rag_simple(query,retriever,llm,top_k=3):
    results = retriever.retrieve(query,top_k=top_k);
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant information found in the documents."
    
    # Generate the answer using groq llm
    prompt = f"""Use the following context to answer the question:\n\n
        Context: {context}\n\n
        Question: {query}\n\n
        Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content.strip();



In [48]:
answer = rag_simple("What is the role of Zayed Shah Syed at cognizant?",retriever=rag_retriever,llm=llm,top_k=3)
print(answer)

Retrieving documents for query: 'What is the role of Zayed Shah Syed at cognizant?' with top_k=3...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.30it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 3 relevant documents.


The role of Zayed Shah Syed at Cognizant is an Intern, specifically a Programmer Analyst Trainee / Programmer Analyst.


## Enhanced Features

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """ 
    Advanced RAG function with enhanced retrieval and response generation
    """
    results = retriever.retrieve(query, top_k=top_k,score_threshold=min_score)
    if not results:
        return {'answer':'No context found','sources':[],'confidence':0.0,'context':''}
    
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page','unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:200] + '...'
        } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    prompt = f"""Use the following context to answer the question:\n\n
    Context: {context}\n\n
    Question: {query}\n\n
    Answer:"""
    response = llm.invoke([prompt.format(context =context,query=query)])
    output = {
        'answer': response.content.strip(),
        'sources': sources,
        'confidence': confidence,
    }
    if return_context:
        output['context'] = context
    return output
    

In [60]:
result = rag_advanced("My Internship starts on March 17th and ends on July 10th. How much total money will I earn till end of internship if I receive 13,500 per month? I will be taking 6 days of leave in May, for which stipend will be deducted", retriever=rag_retriever, llm=llm, top_k=5, min_score=0.2, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Confidence: {result['confidence']:.3f}")
print("Sources:",result['sources'])
print("Context:", result['context'][:300])

Retrieving documents for query: 'My Internship starts on March 17th and ends on July 10th. How much total money will I earn till end of internship if I receive 13,500 per month? I will be taking 6 days of leave in May, for which stipend will be deducted' with top_k=5...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.83it/s]

Embeddings generated successfully with shape: (1, 384).
Retrieved 5 relevant documents.


Answer: To calculate the total amount of money you will earn, we need to calculate the total number of months you will be working and then subtract the stipend for the 6 days of leave in May.

Your internship starts on March 17th and ends on July 10th. This means you will be working for 4 months (March, April, May, and June).

However, since you will be taking 6 days of leave in May, you will not be eligible for the full stipend for that month. Assuming a 30-day month, you will be eligible for 24 days of stipend in May.

The stipend for May will be calculated as follows:

- Total stipend for May: 13,500
- Stipend for 6 days of leave: 13,500 / 30 * 6 = 2,700
- Eligible stipend for May: 13,500 - 2,700 = 10,800

Now, let's calculate the total amount of money you will earn:

- Stipend for March: 13,500
- Stipend for April: 13,500
- Eligible stipend for May: 10,800
- Stipend for June: 13,500

Total amount of money you will earn: 13,500 + 13,500 + 10,800 + 13,500 = 51,300

So, you will earn 